# Fine-Tuning Kırma Testleri (4 Test)
**Konu:** Hugging Face LLM Course, Bölüm 3: BERT ile MRPC (iki cümle aynı anlamda mı?) fine-tuning

## Fikir
Önce **doğru çalışan** eğitimi yapıp sonucu kaydediyoruz (**kontrol**). Sonra kodda **tek bir şeyi bozuyoruz** ve ne olduğuna bakıyoruz. İki ihtimal var:

- **Gürültülü çöküş:** Kod hata verir. Kanıt = hata mesajı. (Fark edilir, tehlikesi az.)
- **Sessiz bozulma:** Kod hatasız çalışır ama sonuç kötüdür. Kanıt = kontrol ile sayı farkı. (Fark edilmez, asıl tehlike bu.)

## 4 test
| # | Ne bozuyoruz? | Kurs bölümü | Tür |
|---|---------------|-------------|-----|
| 1 | Gereksiz sütunları atmayı unutuyoruz (`remove_columns`) | §2, §4 | gürültülü |
| 2 | Gradyan sıfırlamayı siliyoruz (`zero_grad`) | §4 | sessiz |
| 3 | Learning rate'i 200 kat büyütüyoruz (5e-5 → 1e-2) | §5 | sessiz |
| 4 | Modele "kelimeleri aynı, anlamı farklı" cümleler veriyoruz | §3 modeli | sessiz |

## Ayarlar (kursun §4 döngüsüyle aynı)
Batch 8, AdamW (lr 5e-5), doğrusal lr scheduler, 3 epoch (1377 adım), seed 42. Her testte yalnızca bozulan şey değişir.

## Çalıştırma
**Runtime → Change runtime type → GPU.** Toplam yaklaşık 10-15 dk (3 eğitim). Hızlı deneme için `QUICK = True` yap; bu modda sonuçlar kursla kıyaslanmamalı.

> Sonuçlar GPU ve çalıştırmaya göre biraz değişebilir. Anlatırken ekrandaki **gerçek** çıktıyı söyle; beklenen sonuç tutmazsa onu da bulgu olarak anlat.

In [ ]:
!pip install -q -U datasets accelerate scikit-learn

import torch, numpy as np
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, get_scheduler, set_seed)
from torch.utils.data import DataLoader
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score

QUICK = False     # True: sadece 250 adım (hızlı deneme)
SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Cihaz:", device)

summary = []      # sonda tek tabloda göstereceğiz
def log(test, beklenen, gozlenen):
    summary.append((test, beklenen, str(gozlenen)))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 25.0.1 which is incompatible.
Cihaz: cuda


## Hazırlık: veri ve eğitim fonksiyonu (kurs §2 ve §4)
MRPC: iki cümle veriliyor, model "aynı anlamda mı?" diye karar veriyor.
**Etiket 1 = equivalent (aynı anlam), 0 = not_equivalent (farklı anlam).**

In [ ]:
checkpoint = "bert-base-uncased"
raw = load_dataset("nyu-mll/glue", "mrpc")
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized = raw.map(tokenize_function, batched=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Doğru hazırlık: gereksiz sütunları at, label -> labels, torch formatı
clean = tokenized.remove_columns(["sentence1", "sentence2", "idx"]).rename_column("label", "labels")
clean.set_format("torch")
train_dl = DataLoader(clean["train"], shuffle=True, batch_size=8, collate_fn=collator)
eval_dl = DataLoader(clean["validation"], batch_size=8, collate_fn=collator)

# Referans sayılar: "hiçbir şey öğrenmeden" ne kadar doğru çıkar?
oran_equiv = float(np.mean(raw["validation"]["label"]))
print(f"Doğrulama setinde 'equivalent' oranı : {oran_equiv:.4f}  -> hep 'equivalent' dersen accuracy bu")
print(f"Doğrulama setinde 'not_equivalent'   : {1 - oran_equiv:.4f}  -> hep 'not_equivalent' dersen accuracy bu")

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  649kB            

mrpc/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 75.7kB            

mrpc/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

Doğrulama setinde 'equivalent' oranı : 0.6838  -> hep 'equivalent' dersen accuracy bu
Doğrulama setinde 'not_equivalent'   : 0.3162  -> hep 'not_equivalent' dersen accuracy bu


In [ ]:
NUM_EPOCHS = 3
MAX_STEPS = 250 if QUICK else None

def train_loop(lr=5e-5, zero_grad=True):
    set_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2).to(device)
    optimizer = AdamW(model.parameters(), lr=lr)
    num_steps = NUM_EPOCHS * len(train_dl)
    scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_steps)
    model.train()
    losses, step = [], 0
    for epoch in range(NUM_EPOCHS):
        for batch in train_dl:
            batch = {k: v.to(device) for k, v in batch.items()}
            loss = model(**batch).loss
            loss.backward()
            optimizer.step()
            scheduler.step()
            if zero_grad:                 # <-- Test 2'de zero_grad=False yapıp bunu kapatıyoruz
                optimizer.zero_grad()
            losses.append(loss.item())
            step += 1
            if MAX_STEPS and step >= MAX_STEPS:
                return model, losses
    return model, losses

def evaluate_model(model):
    model.eval()
    preds, refs = [], []
    for batch in eval_dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            logits = model(**batch).logits
        preds += logits.argmax(-1).tolist()
        refs += batch["labels"].tolist()
    preds, refs = np.array(preds), np.array(refs)
    return {"accuracy": round(float(accuracy_score(refs, preds)), 4),
            "equiv_orani": round(float(preds.mean()), 3)}   # model örneklerin kaçına 'equivalent' dedi?

def son_loss(losses, n=50):
    return round(float(np.mean(losses[-n:])), 4)

## KONTROL: doğru ayarlarla eğitim
Tüm testler bu sonuçla karşılaştırılacak. Beklenen: accuracy yaklaşık 0.84-0.86.
`equiv_orani`: modelin "aynı anlam" dediği örneklerin oranı. Gerçek oran 0.68 civarı olduğu için sağlıklı bir model buna yakın çıkar.

In [ ]:
model_ok, loss_ok = train_loop()
res_ok = evaluate_model(model_ok)
print("KONTROL ->", res_ok, "| son loss:", son_loss(loss_ok))
log("Kontrol", "accuracy ≈ 0.84-0.86", f"{res_ok} | loss {son_loss(loss_ok)}")

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


KONTROL -> {'accuracy': 0.8554, 'equiv_orani': 0.706} | son loss: 0.0618


## Test 1: Gereksiz sütunları atmayı unuttuk (gürültülü çöküş)
- **Ne bozuyoruz?** `remove_columns` adımını atlıyoruz. `sentence1` ve `sentence2` metin olarak kalıyor.
- **Ne bekliyoruz?** Hata. Model sayı ister, metni tensöre çeviremez.
- **Kanıt:** Hata mesajı. (`Trainer` bunu bizim yerimize yapar; elle yazınca biz unutabiliriz.)

In [ ]:
bad_dl = DataLoader(tokenized["train"].with_format("torch"), batch_size=8, collate_fn=collator)
try:
    next(iter(bad_dl))
    print("Hata çıkmadı (beklenmedik).")
    log("Test 1: sütun atılmadı", "hata verir", "HATA VERMEDİ")
except Exception as e:
    print("ÇÖKTÜ ->", type(e).__name__)
    print(str(e)[:300])
    log("Test 1: sütun atılmadı", "hata verir", type(e).__name__)

ÇÖKTÜ -> ValueError
Unable to create tensor, you should probably activate truncation and/or padding with 'padding=True' 'truncation=True' to have batched tensors with the same length. Perhaps your features (`sentence1` in this case) have excessive nesting (inputs type `list` where type `int` is expected).


## Test 2: Gradyan sıfırlamayı sildik (sessiz bozulma)
- **Ne bozuyoruz?** `optimizer.zero_grad()` satırını siliyoruz. Gradyanlar her adımda üst üste birikiyor.
- **Ne bekliyoruz?** Hata yok. Ama model kötü öğrenir: loss yüksek kalır, accuracy düşer. (AdamW bu hataya kısmen dayanıklı olabilir; öyle çıkarsa bu da bulgudur.)
- **Kanıt:** Kontrol ile accuracy ve loss farkı.

In [ ]:
model_nz, loss_nz = train_loop(zero_grad=False)
res_nz = evaluate_model(model_nz)
print(f"KONTROL -> {res_ok} | loss {son_loss(loss_ok)}")
print(f"BOZUK   -> {res_nz} | loss {son_loss(loss_nz)}")
print(f"Accuracy farkı: {res_ok['accuracy'] - res_nz['accuracy']:+.4f}")
log("Test 2: zero_grad yok", "accuracy düşer, loss yükselir",
    f"acc {res_ok['accuracy']} -> {res_nz['accuracy']} | loss {son_loss(loss_ok)} -> {son_loss(loss_nz)} | equiv oranı {res_nz['equiv_orani']}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


KONTROL -> {'accuracy': 0.8554, 'equiv_orani': 0.706} | loss 0.0618
BOZUK   -> {'accuracy': 0.6838, 'equiv_orani': 1.0} | loss 0.8083
Accuracy farkı: +0.1716


## Test 3: Learning rate'i 200 kat büyüttük (sessiz bozulma)
- **Ne bozuyoruz?** lr: `5e-5` yerine `1e-2`. Adımlar o kadar büyük ki model öğrendiğini sürekli bozuyor.
- **Ne bekliyoruz?** Hata yok. Model öğrenemez; muhtemelen hep aynı cevabı verir.
- **Kanıt:** `equiv_orani` 0 veya 1'e yakınsa model tek cevaba çökmüştür. Accuracy'yi yukarıdaki iki referans sayıyla karşılaştırıyoruz.

In [ ]:
model_hl, loss_hl = train_loop(lr=1e-2)
res_hl = evaluate_model(model_hl)
print(f"KONTROL (lr=5e-5) -> {res_ok}")
print(f"BOZUK   (lr=1e-2) -> {res_hl} | loss {son_loss(loss_hl)}")
print(f"Referans: hep 'equivalent' = {oran_equiv:.4f} | hep 'not_equivalent' = {1 - oran_equiv:.4f}")
log("Test 3: lr=1e-2", "model tek cevaba çöker",
    f"acc {res_hl['accuracy']} | equiv oranı {res_hl['equiv_orani']} | loss {son_loss(loss_hl)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


KONTROL (lr=5e-5) -> {'accuracy': 0.8554, 'equiv_orani': 0.706}
BOZUK   (lr=1e-2) -> {'accuracy': 0.6838, 'equiv_orani': 1.0} | loss 0.6583
Referans: hep 'equivalent' = 0.6838 | hep 'not_equivalent' = 0.3162


## Test 4: Kelimeleri aynı, anlamı farklı cümleler (model zafiyeti)
- **Ne yapıyoruz?** Eğitilmiş modele, anlamı farklı olduğu halde neredeyse aynı kelimelerden oluşan çiftler veriyoruz (sıra değişimi, olumsuzlama, sayı değişimi). Doğru cevap hepsinde **not_equivalent**.
- **Kontrol çiftleri:** Gerçekten aynı anlama gelen bir çift (doğru cevap `equivalent`) ve alakasız bir çift (doğru cevap `not_equivalent`). Bunlar, modelin "her şeye equivalent diyen" bir model olup olmadığını ayırt etmemizi sağlar.
- **Ne bekliyoruz?** Model anlama değil kelime benzerliğine bakıyorsa zor çiftlerde yanılır, kontrol çiftlerinde doğru bilir.
- **Kanıt:** Olasılıklar. Yüksek `equivalent` olasılığı = model "aynı anlam" konusunda emin, yanlış olsa bile.

In [ ]:
def predict(s1, s2):
    model_ok.eval()
    enc = tokenizer(s1, s2, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        p = model_ok(**enc).logits.softmax(-1)[0]
    return {"not_equivalent": round(p[0].item(), 3), "equivalent": round(p[1].item(), 3)}

# (cümle 1, cümle 2, doğru etiket, grup)
pairs = [
    ("Flights from New York to Florida are cheap.", "Flights from Florida to New York are cheap.", 0, "zor"),
    ("The company raised its prices.", "The company did not raise its prices.", 0, "zor"),
    ("Revenue grew to 5 million dollars.", "Revenue grew to 50 million dollars.", 0, "zor"),
    ("The company raised its prices.", "The firm increased its prices.", 1, "kontrol: gerçek paraphrase"),
    ("The weather is lovely today.", "Stocks fell sharply on Monday.", 0, "kontrol: alakasız"),
]

skor = {}
for a, b, gold, grup in pairs:
    r = predict(a, b)
    tahmin = int(r["equivalent"] > 0.5)
    dogru = tahmin == gold
    g = grup.split(":")[0]
    skor.setdefault(g, [0, 0]); skor[g][0] += int(dogru); skor[g][1] += 1
    print(("DOĞRU  " if dogru else "YANLIŞ "), f"[{grup}]", r)
    print("        ", a, "->", b)

ozet = " | ".join(f"{g}: {d}/{n} doğru" for g, (d, n) in skor.items())
print("\n", ozet)
log("Test 4: kelime sırası/olumsuzlama/sayı", "zor çiftlerde yanılır, kontrollerde doğru bilir", ozet)

YANLIŞ  [zor] {'not_equivalent': 0.003, 'equivalent': 0.997}
         Flights from New York to Florida are cheap. -> Flights from Florida to New York are cheap.
YANLIŞ  [zor] {'not_equivalent': 0.003, 'equivalent': 0.997}
         The company raised its prices. -> The company did not raise its prices.
YANLIŞ  [zor] {'not_equivalent': 0.075, 'equivalent': 0.925}
         Revenue grew to 5 million dollars. -> Revenue grew to 50 million dollars.
DOĞRU   [kontrol: gerçek paraphrase] {'not_equivalent': 0.003, 'equivalent': 0.997}
         The company raised its prices. -> The firm increased its prices.
DOĞRU   [kontrol: alakasız] {'not_equivalent': 0.992, 'equivalent': 0.008}
         The weather is lovely today. -> Stocks fell sharply on Monday.

 zor: 0/3 doğru | kontrol: 2/2 doğru


## Özet tablo (son slayt için)

In [ ]:
for test, beklenen, gozlenen in summary:
    print(f"{test}\n   Beklenen : {beklenen}\n   Gözlenen : {gozlenen}\n")

Kontrol
   Beklenen : accuracy ≈ 0.84-0.86
   Gözlenen : {'accuracy': 0.8554, 'equiv_orani': 0.706} | loss 0.0618

Test 1: sütun atılmadı
   Beklenen : hata verir
   Gözlenen : ValueError

Test 2: zero_grad yok
   Beklenen : accuracy düşer, loss yükselir
   Gözlenen : acc 0.8554 -> 0.6838 | loss 0.0618 -> 0.8083 | equiv oranı 1.0

Test 3: lr=1e-2
   Beklenen : model tek cevaba çöker
   Gözlenen : acc 0.6838 | equiv oranı 1.0 | loss 0.6583

Test 4: kelime sırası/olumsuzlama/sayı
   Beklenen : zor çiftlerde yanılır, kontrollerde doğru bilir
   Gözlenen : zor: 0/3 doğru | kontrol: 2/2 doğru



## Sonuç ve sınırlar
- **Kod hatası** elle yazılan döngüde ya çöküş ya da sessiz bozulma üretir. Çöküş fark edilir, sessiz bozulma fark edilmez; bu yüzden eğitim sonunda sonucu her zaman **referans sayılarla** (hep aynı cevabı vermenin accuracy'si) karşılaştırmak gerekir.
- **Model hatası:** İyi accuracy, modelin anlamı öğrendiği anlamına gelmez. Zor örneklerle ayrıca sınamak gerekir.
- **Sınırlar:** Tek seed, 408 doğrulama örneği (bir örnek ≈ %0.245), Test 4'te yalnızca 5 çift. Sonuçlar gösterim amaçlıdır, istatistiksel iddia değildir.